In [1]:
# pipeline base params
table_name = "payroll_fact_gold"

StatementMeta(, 28157b77-dd98-4cc8-9f70-64cd2cdbfdf3, 3, Finished, Available, Finished, False)

In [2]:
from pyspark.sql import functions as F

# Load Silver Payroll
df = spark.read.format("delta").load(
    "abfss://dev_Silver@onelake.dfs.fabric.microsoft.com/lh_nyc_payroll_silver.Lakehouse/Tables/payroll_data_silver")
display(df)

StatementMeta(, 28157b77-dd98-4cc8-9f70-64cd2cdbfdf3, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5bb47188-369a-44dc-bab0-8efd78072f1c)

In [3]:
# Register the dataframe as SQL View 
df.createOrReplaceTempView("silver_payroll")

StatementMeta(, 28157b77-dd98-4cc8-9f70-64cd2cdbfdf3, 5, Finished, Available, Finished, False)

In [4]:
# Create Gold Fact Table with SQL
spark.sql(f"""
CREATE OR REPLACE TABLE {table_name}
USING DELTA
AS
SELECT
    fiscalyear,
    agencyid,
    titlecode,

    /* Data Quality Flag */
    CASE
        WHEN fiscalyear < 2020 THEN true
        ELSE false
    END AS fiscalyear_outlier_flag,

    /* Create Payroll Fact Summary */
    SUM(totalotpaid) AS total_overtime,
    AVG(basesalary) AS avg_salary,

    SUM(
        regulargrosspaid +
        totalotpaid +
        totalotherpay
    ) AS total_payroll,

    /* Derive Overtime Percentage KPI */
    CASE
        WHEN SUM(
            regulargrosspaid +
            totalotpaid +
            totalotherpay
        ) > 0
        THEN
            (SUM(totalotpaid) /
             SUM(
                 regulargrosspaid +
                 totalotpaid +
                 totalotherpay
             )) * 100
        ELSE 0
    END AS percentage_overtime

FROM silver_payroll
GROUP BY
    fiscalyear,
    agencyid,
    titlecode
""")

# Query and display the Gold fact table
display(spark.table(table_name))


StatementMeta(, 28157b77-dd98-4cc8-9f70-64cd2cdbfdf3, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a6a044d9-9e3b-4762-81f5-ba3a14f687de)